### Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1]))

from utils.loader import get_loader, FgsmParams, PgdTrainerParams
from utils.notebook import FilenameLoader

### Run Function
Function to run full test on one PGD training eps

In [ ]:
def run_test(file_info: tuple[str, str, str], pgd_eps: float):
    """
    - file_info: tuple of the [checkpoint file, data file, and save directory].
        All should be relative to the repo root
    - pge_eps: epsilon that the PGD adv training should be done on
    """
    ckpt_file, data_file, save_name = file_info
    root = pathlib.Path("defenses/fed/adv-test")
    loader = get_loader(ckpt_file = ckpt_file,
                        data_file = data_file,
                        save_dir = root / f"{save_name}_targeted_{pgd_eps}")
    # Run clean before
    loader.test_clean(filename="before_clean")

    # Adv training
    params = PgdTrainerParams(
        pgd_eps = pgd_eps,
        pgd_iter = 5,
        ratio = 0.5,
        epochs = 5
    )
    loader.train_adv_pgd(params)

    # Run clean after
    loader.test_clean(filename="clean")

    # Run FGSM adv sweep
    for i in range(1, 31):
        params = FgsmParams(eps = float(i/100))
        loader.test_adv_fgsm(params)

    loader.save_model_config()

### Run on a range of eps

In [ ]:
for i in range(1, 16, 1):
    eps = float(i/100)
    run_test(FilenameLoader.rand_pos(), eps)